### Install dependencies

In [1]:
%%sh
pip install boto3 
pip install sagemaker
pip install tensorflow

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


### Setup sagemaker session

In [20]:
import sagemaker

print(sagemaker.__version__)

sess = sagemaker.Session()  # Use the AWS credentials and region configured with the AWS CLI

# This is the SageMaker role you're already using, it will work just fine
role = 'arn:aws:iam::817539924586:role/SageMaker_FullAccess'

INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials


2.168.0


### Download data

In [10]:
import os
import keras
import numpy as np
from keras.datasets import fashion_mnist

(x_train, y_train), (x_val, y_val) = fashion_mnist.load_data()

os.makedirs("./tfdata", exist_ok = True)

np.savez('./tfdata/training', image=x_train, label=y_train)
np.savez('./tfdata/validation', image=x_val, label=y_val)

### Local example

In [11]:
# Train on local data. S3 URIs would work too.
training_input_path   = 'file://tfdata/training.npz'
validation_input_path = 'file://tfdata/validation.npz'

# Store model locally. A S3 URI would work too.
output_path           = 'file://tfmodel/'

In [6]:
from sagemaker.tensorflow import TensorFlow

tf_estimator = TensorFlow(entry_point='mnist_keras_tf20_eager.py',
                          role=role,
                          instance_count=1, 
                          instance_type='local',   # Train on the local CPU ('local_gpu' if it has a GPU)
                          framework_version='2.0', 
                          py_version='py3',
                          hyperparameters={'epochs': 1},
                          output_path=output_path
                         )

In [7]:
# Train! This will pull (once) the SageMaker CPU/GPU container for TensorFlow to your local machine.
# Make sure that Docker is running and that docker-compose is installed

tf_estimator.fit({'training': training_input_path, 'validation': validation_input_path})

INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: tensorflow-training-2023-06-26-10-15-55-528
INFO:sagemaker.local.local_session:Starting training job
INFO:sagemaker.local.image:Using the long-lived AWS credentials found in session
INFO:sagemaker.local.image:docker compose file: 
networks:
  sagemaker-local:
    name: sagemaker-local
services:
  algo-1-q7twq:
    command: train
    container_name: 0gl35gy14a-algo-1-q7twq
    environment:
    - '[Masked]'
    - '[Masked]'
    - '[Masked]'
    - '[Masked]'
    image: 763104351884.dkr.ecr.eu-west-1.amazonaws.com/tensorflow-training:2.0-cpu-py3
    networks:
      sagemaker-local:
        aliases:
        - algo-1-q7twq
    stdin_open: true
    tty: true
    volumes:
    - /tmp/tmpths1sip7/algo-1-q7twq/output/data:/opt/ml/output/data
    - /tmp/tmpths1sip7/algo-1-q7twq/output:/opt/ml/output
    - /tmp/tmpths1sip7/algo-1-q7twq/inp

Container 0gl35gy14a-algo-1-q7twq  Creating
Container 0gl35gy14a-algo-1-q7twq  Created
Attaching to 0gl35gy14a-algo-1-q7twq
0gl35gy14a-algo-1-q7twq  | 2023-06-26 10:15:59,333 sagemaker-training-toolkit INFO     Imported framework sagemaker_tensorflow_container.training
0gl35gy14a-algo-1-q7twq  | 2023-06-26 10:15:59,343 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
0gl35gy14a-algo-1-q7twq  | 2023-06-26 10:15:59,380 botocore.credentials INFO     Found credentials in environment variables.
0gl35gy14a-algo-1-q7twq  | 2023-06-26 10:15:59,762 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
0gl35gy14a-algo-1-q7twq  | 2023-06-26 10:15:59,795 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
0gl35gy14a-algo-1-q7twq  | 2023-06-26 10:15:59,817 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
0gl35gy14a-algo-1-q7twq  | 2023-06-26 10:15:59,830 sagemaker-training

### AWS example

In [22]:
# upload data to S3 if required
response = sess.s3_client.list_objects(Bucket="pg-algo-data", Prefix="tfdata")
if not 'Contents' in response:
    sess.s3_client.upload_file('tfdata/training.npz', "pg-algo-data", "tfdata/training.npz")
    sess.s3_client.upload_file('tfdata/validation.npz', "pg-algo-data", "tfdata/validation.npz")

# Train on local data. S3 URIs would work too.
training_input_path   = 's3://pg-algo-data/tfdata/training.npz'
validation_input_path = 's3://pg-algo-data/tfdata/validation.npz'

# Store model locally. A S3 URI would work too.
output_path           = 's3://pg-algo-data/tfmodel'

In [17]:
from sagemaker.tensorflow import TensorFlow

tf_estimator = TensorFlow(entry_point='mnist_keras_tf20_eager.py',
                          role=role,
                          instance_count=1, 
                          # instance_type='local',   # Train on the local CPU ('local_gpu' if it has a GPU)
                          instance_type='ml.g4dn.xlarge',
                          framework_version='2.0', 
                          py_version='py3',
                          hyperparameters={'epochs': 1},
                          output_path=output_path
                         )

INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials


In [18]:
# Train! This will pull (once) the SageMaker CPU/GPU container for TensorFlow to your local machine.
# Make sure that Docker is running and that docker-compose is installed

tf_estimator.fit({'training': training_input_path, 'validation': validation_input_path})

INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.


Using provided s3_resource


INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: tensorflow-training-2023-06-26-10-25-12-081


2023-06-26 10:25:13 Starting - Starting the training job...
2023-06-26 10:25:28 Starting - Preparing the instances for training......
2023-06-26 10:26:34 Downloading - Downloading input data...
2023-06-26 10:26:54 Training - Downloading the training image......
2023-06-26 10:28:10 Training - Training image download completed. Training in progress.../usr/local/lib/python3.6/site-packages/pandas/compat/__init__.py:117: UserWarning: Could not import the lzma module. Your installed Python is incomplete. Attempting to use lzma compression will result in a RuntimeError.
  warnings.warn(msg)
2023-06-26 10:28:23,165 sagemaker-training-toolkit INFO     Imported framework sagemaker_tensorflow_container.training
2023-06-26 10:28:23,529 sagemaker-training-toolkit INFO     Invoking user script
Training Env:
{
    "additional_framework_parameters": {},
    "channel_input_dirs": {
        "training": "/opt/ml/input/data/training",
        "validation": "/opt/ml/input/data/validation"
    },
    "curr